# Sentinel-1 Band Sources -- Live Validation

This notebook is a **manual** validation of the band-sources work (issue [#348](
https://github.com/sentinel-hub/titiler-openeo/issues/348), ADR
[`docs/adr/0002-band-sources.md`](../adr/0002-band-sources.md)) against a real,
running backend and real Sentinel-1 GRD data -- the same spirit as
`sar_backscatter_rgb.ipynb`, but checking the band-sources pipeline instead of
`sar_backscatter` itself.

It checks three things, in order:

1. **Discovery (increment 1).** The derived band names show up in
   `describe_collection("sentinel-1-grd")`'s `cube:dimensions`.
2. **Production (increment 2).** `vv_noise_lut`/`vh_noise_lut` are actually
   *readable* via `load_collection`, and -- the important check -- their values
   reproduce `sar_backscatter`'s own noise subtraction exactly, computed two
   completely independent ways from the same live scene.
3. **Cost, live.** Reproduces the increment-2 gate measurement (ADR 0002 S4,
   increment 2) -- decomposed (band-source reads) vs. fused (`sar_backscatter`)
   -- against a real backend and a real scene's real GCP count, rather than the
   synthetic scaled fixture the gate itself used.

**This notebook is meant to be reused, not just run once.** Increment 3 adds
`CalibrationBandReader` (five more bands per polarisation); the last section
below describes exactly how to extend the same checks to them once that lands
-- no new notebook needed.

**Requires:** a running local titiler-openeo backend configured against a real
STAC catalogue with credentials (see `.env.cdse`), and the `openeo` Python
client (`pip install openeo`) -- these notebooks run in their own environment,
`openeo` is not a project dependency.

## Import Required Libraries

In [1]:
import time

import matplotlib.pyplot as plt
import numpy as np
import openeo
import rasterio

## Connect to OpenEO Backend

Same pattern as the other notebooks in this folder.

In [2]:
# connection = openeo.connect(
#     url="https://openeo.ds.io"
# ).authenticate_oidc_authorization_code()

connection = openeo.connect(
    url="http://127.0.0.1:8082/"
).authenticate_oidc_authorization_code()

## 1. Check Discovery

`getdimensions` (`stacapi.py`) should advertise every band the registry
describes for `sentinel-1-grd` -- both the one increment 2 makes readable
(`*_noise_lut`) and the five increment 3 will make readable
(`*_sigma0_lut`/`*_beta0_lut`/`*_gamma0_lut`/`*_dn_lut`/
`*_ellipsoid_incidence_angle`), since discovery is allowed to advertise a band
before its reader exists (that is increment 1's whole point -- see ADR 0002
S2.1).

In [3]:
collection = connection.describe_collection("sentinel-1-grd")
spectral = next(
    dim for dim in collection["cube:dimensions"].values() if dim["type"] == "bands"
)
band_names = set(spectral["values"])
print(f"{len(band_names)} bands advertised:")
print(sorted(band_names))

readable_now = {"vv_noise_lut", "vh_noise_lut"}
discovery_only_for_now = {
    f"{pol}_{suffix}"
    for pol in ("vv", "vh")
    for suffix in (
        "sigma0_lut",
        "beta0_lut",
        "gamma0_lut",
        "dn_lut",
        "ellipsoid_incidence_angle",
    )
}
missing = (readable_now | discovery_only_for_now) - band_names
if missing:
    print(f"\nMISSING from discovery: {sorted(missing)}")
else:
    print(
        "\nAll expected derived band names (increment 2 + increment 3) are advertised."
    )

28 bands advertised:
['hh', 'hh_beta0_lut', 'hh_dn_lut', 'hh_ellipsoid_incidence_angle', 'hh_gamma0_lut', 'hh_noise_lut', 'hh_sigma0_lut', 'hv', 'hv_beta0_lut', 'hv_dn_lut', 'hv_ellipsoid_incidence_angle', 'hv_gamma0_lut', 'hv_noise_lut', 'hv_sigma0_lut', 'vh', 'vh_beta0_lut', 'vh_dn_lut', 'vh_ellipsoid_incidence_angle', 'vh_gamma0_lut', 'vh_noise_lut', 'vh_sigma0_lut', 'vv', 'vv_beta0_lut', 'vv_dn_lut', 'vv_ellipsoid_incidence_angle', 'vv_gamma0_lut', 'vv_noise_lut', 'vv_sigma0_lut']

All expected derived band names (increment 2 + increment 3) are advertised.


## 2. Load Sentinel-1 GRD

The AOI below is the same Tokyo Bay acquisition `sar_backscatter_rgb.ipynb`
uses: open water, dense urban/industrial structures, and hills inland, so
noise and backscatter aren't uniform across the scene -- worth having variety
when eyeballing plots later.

Same two traps as that notebook: `temporal_extent` must be ISO 8601 with the
`T`, and a narrow, sub-minute window is the reliable way to pin exactly one
acquisition (Sentinel-1 footprints are slanted parallelograms, so a wider
window risking >1 item defeats the pixel-for-pixel comparison below --
`sar_backscatter` itself refuses to calibrate a mosaicked, >1-item slice, and
band sources currently don't get exercised against that case here either).

`GRID_SIZE` is fixed and reused for every request in this notebook -- the
cross-check in section 3 only means something if every cube lands on the
*identical* destination grid.

In [4]:
# Tokyo Bay AOI (same acquisition as sar_backscatter_rgb.ipynb)
spatial_extent = {"west": 139.5, "south": 35.2, "east": 140.2, "north": 35.8}
temporal_extent = [
    "2026-07-08T20:42:55Z",
    "2026-07-08T20:43:20Z",
]  # adjust to a real S1 pass over the AOI

GRID_SIZE = 512
POLARISATIONS = ["vv", "vh"]

## 3. Load Two Cubes on the Same Grid, Cross-Validate

Two `load_collection` calls, identical in every parameter except `bands`:

* **`dn_plus_derived`** -- the DN bands plus `<pol>_noise_lut` for each, read
  through the band-sources path this notebook is validating.
* **`fused_null`** -- the DN bands alone, then `sar_backscatter(coefficient=
  None, noise_removal=True)`. `coefficient=None` is openEO's `null` --
  `sar_backscatter` returns `DN^2` uncalibrated, minus the noise LUT it
  evaluates *internally*, clamped at 0 (`calibration.py:51-59`). Crucially,
  this needs no calibration LUT at all, so it is checkable today, without
  waiting for increment 3.

If `vv_noise_lut`/`vh_noise_lut` are correct, `max(DN^2 - noise_lut, 0)`
computed by hand from the first cube's bands must equal the second cube's
output exactly -- two independent computations (one server-side inside
`sar_backscatter`, one by hand in this notebook from bands read a completely
different way) landing on the same numbers is a much stronger check than
either alone.

`width=`/`height=` are backend-specific `load_collection` parameters (not
part of the openEO spec), so they go through `datacube_from_process` rather
than the client's `.load_collection()` convenience wrapper -- the same
pattern `evi.ipynb` uses for `width=None`.

In [5]:
def load_s1(bands, width=GRID_SIZE, height=GRID_SIZE):
    cube = connection.datacube_from_process(
        "load_collection",
        id="sentinel-1-grd",
        spatial_extent=spatial_extent,
        temporal_extent=temporal_extent,
        bands=bands,
        width=width,
        height=height,
    )
    return cube.reduce_dimension(dimension="t", reducer="firstpixel")


noise_lut_bands = [f"{pol}_noise_lut" for pol in POLARISATIONS]

dn_plus_derived = load_s1(POLARISATIONS + noise_lut_bands)
dn_plus_derived.download("band_sources_dn_plus_noise.tif", format="GTiff")

fused_null = load_s1(POLARISATIONS)
fused_null = fused_null.process(
    "sar_backscatter", data=fused_null, coefficient=None, noise_removal=True
)
fused_null.download("band_sources_fused_null.tif", format="GTiff")

with rasterio.open("band_sources_dn_plus_noise.tif") as src:
    print("dn_plus_derived band order:", src.descriptions)
with rasterio.open("band_sources_fused_null.tif") as src:
    print("fused_null band order:      ", src.descriptions)

OpenEoApiError: [400] ProcessParameterInvalid: sar_backscatter requires `data` to come from load_collection or load_stac, which carry the STAC metadata it needs; the slice at 1900-01-01 00:00:00 has no source-item metadata (a stack built from in-memory images cannot be calibrated)

In [ ]:
with rasterio.open("band_sources_dn_plus_noise.tif") as src:
    derived = src.read(masked=True).astype("float64")
with rasterio.open("band_sources_fused_null.tif") as src:
    fused = src.read(masked=True).astype("float64")

n = len(POLARISATIONS)
dn = derived[:n]
noise_lut = derived[n:]

for i, pol in enumerate(POLARISATIONS):
    manual = np.ma.maximum(dn[i] ** 2 - noise_lut[i], 0.0)
    valid = ~(manual.mask | fused[i].mask) if hasattr(manual, "mask") else slice(None)
    diff = np.abs(manual.filled(0) - fused[i].filled(0))[valid]
    if diff.size == 0:
        print(f"{pol}: no overlapping valid pixels -- widen the AOI/time window")
        continue
    match = np.allclose(
        manual.filled(0)[valid], fused[i].filled(0)[valid], rtol=1e-4, atol=1e-3
    )
    print(
        f"{pol}: max abs diff = {diff.max():.6g}, mean = {diff.mean():.6g}, "
        f"{valid.sum() if hasattr(valid, 'sum') else n}px compared -- "
        f"{'MATCH' if match else 'MISMATCH -- investigate before trusting the pipeline'}"
    )

## 4. Visual Sanity Check

The noise LUT should look like a smooth, low-frequency field (ESA's LUTs are
sampled every ~1500 pixels in range and vary smoothly -- ADR
[0001](../adr/0001-sar-backscatter.md) S1.6d) -- not noisy, not blocky, and
not a constant (a flat image at this stage usually means the destination grid
missed the item's real footprint -- see `sar_backscatter_rgb.ipynb`'s note on
slanted parallelograms and the `mask` band).

In [ ]:
fig, axes = plt.subplots(1, len(POLARISATIONS), figsize=(6 * len(POLARISATIONS), 5))
if len(POLARISATIONS) == 1:
    axes = [axes]
for ax, pol, lut in zip(axes, POLARISATIONS, noise_lut):
    im = ax.imshow(lut, cmap="viridis")
    ax.set_title(f"{pol}_noise_lut")
    ax.axis("off")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

## 5. Live Cost Check: Decomposed vs. Fused

Reproduces the increment-2 gate (ADR 0002 S4) against this backend's real
scene instead of the synthetic 440-GCP polar fixture the gate script used
locally.

**Read this number correctly.** The comparison that matters is *decomposed
vs. fused*, not *with-LUT vs. without*: both `dn_plus_derived` and
`fused_null` pay the same TPS inverse-map fit (`geocode.build_inverse_map`)
exactly once each. An early version of the local gate measurement compared
against a bare DN-only read and found a 33x-233x blowup -- that comparison
was wrong, and measured the cost of wanting a LUT at all, not the cost of
decomposition. What this cell checks is whether **reading the LUT as a
separate band costs more than `sar_backscatter` already costs today** -- the
local measurement said no (1.00x at both 256^2 and 1024^2). This is that same
check, live.

Expect the *ratio* to land near 1.0 regardless of network/server overhead
(both requests pay it), even though the *absolute* seconds here will differ
from the local synthetic-fixture numbers -- this scene's real GCP count
almost certainly differs from the fixture's 440.

In [ ]:
SIZES = [256, 512, 1024]
results = []

for size in SIZES:
    decomposed = load_s1(POLARISATIONS + noise_lut_bands, width=size, height=size)
    fused = load_s1(POLARISATIONS, width=size, height=size)
    fused = fused.process(
        "sar_backscatter", data=fused, coefficient=None, noise_removal=True
    )

    t0 = time.perf_counter()
    decomposed.download(f"_tmp_decomposed_{size}.tif", format="GTiff")
    t_decomposed = time.perf_counter() - t0

    t0 = time.perf_counter()
    fused.download(f"_tmp_fused_{size}.tif", format="GTiff")
    t_fused = time.perf_counter() - t0

    ratio = t_decomposed / t_fused if t_fused else float("nan")
    results.append((size, t_decomposed, t_fused, ratio))
    print(
        f"{size:>5}px  decomposed={t_decomposed:7.2f}s  fused={t_fused:7.2f}s  ratio={ratio:5.2f}"
    )

In [ ]:
sizes, t_decomposed, t_fused, _ = zip(*results)
plt.figure(figsize=(7, 5))
plt.plot(sizes, t_decomposed, "o-", label="decomposed (band source)")
plt.plot(sizes, t_fused, "s-", label="fused (sar_backscatter)")
plt.xlabel("grid size (pixels, square)")
plt.ylabel("wall time (s)")
plt.title("Live gate: decomposed vs. fused, per-tile time")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Extending This Notebook for Increment 3

Increment 3 ships `CalibrationBandReader`, adding five more bands per
polarisation: `<pol>_sigma0_lut`, `<pol>_beta0_lut`, `<pol>_gamma0_lut`,
`<pol>_dn_lut`, `<pol>_ellipsoid_incidence_angle`. To validate it here once it
lands, without writing a new notebook:

1. **Section 3's cross-check generalises directly.** Add the calibration band
   you want (e.g. `f"{pol}_sigma0_lut"`) to `dn_plus_derived`'s band list, and
   compare against `sar_backscatter(coefficient="sigma0-ellipsoid",
   noise_removal=True)` on the DN-only cube. The manual reproduction is
   `calibration.py`'s own formula:
   `sigma0 = max(DN^2 - noise_lut, 0) / sigma0_lut**2` -- swap in `beta0_lut`/
   `gamma0_lut` for the other coefficients.
2. **`ellipsoid_incidence_angle` has its own oracle**, independent of DN
   entirely: `sar_backscatter(..., ellipsoid_incidence_angle=True)` appends a
   band that must equal `<pol>_ellipsoid_incidence_angle` read directly --
   no arithmetic needed, just a direct pixel comparison.
3. **Re-run the section 5 timing cell with the extra bands added.** ADR 0002's
   risk log flags that each *additional* LUT band sharing a polarisation
   currently rebuilds its own inverse map (no cache yet, by design -- "do not
   add a cache speculatively" until the duplication is real). Increment 3 is
   exactly where it becomes real: watch whether the decomposed/fused ratio
   stays near 1.0 as more calibration bands are added to one request, or
   starts climbing with band count -- that is the signal for whether the
   inverse-map memo mentioned in the ADR's risks is worth building sooner
   rather than later.